Notebook Exploracion Lending Club 

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("lending_club_exploration")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark

In [ ]:
# Leemos el CSV comprimido. Spark maneja .gz nativamente.
# header=True: primera fila son nombres de columnas
# inferSchema=True: Spark infiere tipos para poder explorar los tipos de datos
df = spark.read.csv(
    "../data/raw/accepted_2007_to_2018Q4.csv.gz",
    header=True,
    inferSchema=True,
)

In [ ]:
df.printSchema()

In [ ]:
# Cantidad total de columnas y filas
print(f"Columnas: {len(df.columns)}")
print(f"Filas: {df.count():,}")

Missing Rate

In [ ]:
from pyspark.sql import functions as F

# Contamos nulls por cada columna y calculamos porcentaje
total_rows = df.count()

# Para cada columna, contamos cuántos son null
missing_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).collect()[0].asDict()

# Convertimos a lista de tuplas (columna, count_nulls, pct_nulls)
missing_stats = [
    (col, count, round(count / total_rows * 100, 2))
    for col, count in missing_counts.items()
]

# Ordenamos por porcentaje descendente
missing_stats.sort(key=lambda x: x[2], reverse=True)

# Mostramos top 30 columnas con más nulls
print(f"{'Columna':<40} {'Nulls':>12} {'% Missing':>10}")
print("-" * 65)
for col, count, pct in missing_stats[:30]:
    print(f"{col:<40} {count:>12,} {pct:>9}%")

In [ ]:
# Missing rate rango 30-60
print(f"{'Columna':<45} {'Nulls':>12} {'% Missing':>10}")
print("-" * 70)
for col, count, pct in missing_stats[30:60]:
    print(f"{col:<45} {count:>12,} {pct:>9}%")

In [ ]:
# Missing rate rango 60-100 (columnas mas limpias)
print(f"{'Columna':<45} {'Nulls':>12} {'% Missing':>10}")
print("-" * 70)
for col, count, pct in missing_stats[60:100]:
    print(f"{col:<45} {count:>12,} {pct:>9}%")

In [ ]:
# Lista propuesta de columnas para bronze
selected_columns = [
    # Identificadores
    "id",
    # Términos del préstamo
    "loan_amnt", "funded_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    # Info del deudor
    "emp_title", "emp_length", "home_ownership", "annual_inc", "verification_status",
    "dti", "application_type",
    # Origen y propósito
    "purpose", "title", "zip_code", "addr_state",
    # Fechas clave
    "issue_d", "last_pymnt_d", "earliest_cr_line",
    # Estado y performance
    "loan_status", "pymnt_plan", "out_prncp", "total_pymnt", "total_rec_prncp",
    "total_rec_int", "total_rec_late_fee", "recoveries",
    # Metricas crediticias
    "fico_range_low", "fico_range_high", "revol_util",
]